# 웹 데이터 수집(크롤링)

웹 페이지의 HTML을 요청하고 필요한 정보를 추출해 분석 가능한 표로 만드는 과정을 실습합니다.

이번 노트북의 학습 목표:

- HTTP 요청과 HTML 구조의 기본 이해
- `requests`와 `BeautifulSoup`으로 웹 페이지 읽기
- 상품명, 가격, 평점을 추출해 pandas DataFrame으로 정리하기
- 수집한 데이터를 CSV 파일로 저장하기

실습은 웹 스크래핑 연습용 공개 사이트인 Books to Scrape를 사용합니다. 실제 사이트를 수집할 때는 이용 약관과 `robots.txt`를 확인하고, 개인정보나 로그인이 필요한 정보를 수집하지 마세요. 서버에 부담을 주지 않도록 요청 횟수와 속도도 제한해야 합니다.

## 1. HTML 구조 살펴보기

웹 페이지는 태그로 구성된 HTML 문서입니다. 크롤링할 때는 브라우저의 개발자 도구로 원하는 데이터가 어떤 태그와 클래스에 들어 있는지 먼저 확인합니다.

Books to Scrape의 상품 카드는 `article.product_pod` 요소 하나에 해당합니다. 카드 안에서 제목은 `h3 a`, 가격은 `.price_color`, 평점은 `.star-rating` 요소에서 찾을 수 있습니다.

In [ ]:
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

url = 'https://books.toscrape.com/'
response = requests.get(url, timeout=15)
response.raise_for_status()

print('HTTP 상태 코드:', response.status_code)
print('페이지 제목:', BeautifulSoup(response.text, 'html.parser').title.get_text(strip=True))

`requests.get()`은 페이지를 요청하고, `timeout`은 응답을 기다리는 최대 시간(초)을 지정합니다. `raise_for_status()`는 요청이 실패한 경우 오류를 발생시켜 문제가 있는데도 수집이 성공한 것처럼 진행되지 않게 합니다.

## 2. 필요한 데이터 추출하기

`BeautifulSoup`으로 HTML을 파싱한 뒤 CSS 선택자로 상품 카드와 각 필드를 찾습니다. 제목 링크는 상대 경로일 수 있으므로 `urljoin()`으로 전체 주소를 만듭니다. 페이지 구조가 예상과 다르면 명시적으로 오류를 내도록 확인합니다.

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')
product_cards = soup.select('article.product_pod')

if not product_cards:
    raise ValueError('상품 카드(article.product_pod)를 찾지 못했습니다. 페이지 구조가 바뀌었는지 확인하세요.')

rating_values = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
products = []

for card in product_cards:
    title_link = card.select_one('h3 a')
    price_element = card.select_one('.price_color')
    rating_element = card.select_one('.star-rating')

    if title_link is None or price_element is None or rating_element is None:
        raise ValueError('상품 카드에서 제목, 가격 또는 평점을 찾지 못했습니다.')

    rating_classes = rating_element.get('class', [])
    rating_names = [name for name in rating_classes if name in rating_values]
    if len(rating_names) != 1:
        raise ValueError(f'평점 클래스를 해석할 수 없습니다: {rating_classes}')

    products.append({
        'title': title_link.get('title', title_link.get_text(strip=True)),
        'price_gbp': float(price_element.get_text(strip=True).replace('£', '')),
        'rating': rating_values[rating_names[0]],
        'url': urljoin(url, title_link.get('href', '')),
    })

books_df = pd.DataFrame(products)
books_df.head()

## 3. 수집 결과 확인 및 저장

수집 직후에는 행 수와 결측치를 확인해 데이터가 예상대로 만들어졌는지 점검합니다. 이후 pandas의 `to_csv()`로 저장하면 다른 분석 노트북에서도 사용할 수 있습니다.

In [ ]:
print('수집한 상품 수:', len(books_df))
print('\n컬럼별 결측치:')
print(books_df.isna().sum())
print('\n평점별 상품 수:')
print(books_df['rating'].value_counts().sort_index())

output_path = 'books_catalog.csv'
books_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'CSV 저장 완료: {output_path}')

## 4. 정리와 연습 문제

- `requests`로 웹 페이지를 요청하고 응답 상태를 확인했습니다.
- `BeautifulSoup`과 CSS 선택자로 HTML에서 필요한 필드를 추출했습니다.
- pandas DataFrame으로 정리하고 CSV 파일로 저장했습니다.

연습해 보기:

1. 수집 결과에서 평점이 4 이상인 상품만 골라 보세요.
2. 가격이 가장 낮은 상품 5개를 찾아 보세요.
3. Books to Scrape의 다음 페이지를 한 번만 추가로 요청해 같은 방식으로 합쳐 보세요. 여러 페이지를 자동 순회하기 전에는 사이트의 이용 조건과 요청 간격을 먼저 확인하세요.